In [0]:
df_online_silver = spark.read.table("workspace.superstore.silver_online_superstore")
df_online_silver.createOrReplaceTempView("silver_superstore_view")

In [0]:
#location dimention
dim_location = spark.sql(""" WITH dim_location AS (      
SELECT DISTINCT
  postal_code,
  country,
  region,
  state,
  city
FROM
  silver_superstore_view
WHERE 
  postal_code IS NOT NULL
)
SELECT
  *
FROM
  dim_location
  """)



In [0]:
#writing to dim_location

(dim_location.write
  .format("delta")
  .mode("overwrite")
  .option("overwriteSchema", "true")
  .saveAsTable("workspace.superstore.gold_dim_location"))

In [0]:
#customer dimention
dim_customer = spark.sql(""" WITH dim_customer AS (      
SELECT DISTINCT
  customer_id,
  customer_name,
  segment
FROM
  silver_superstore_view
WHERE 
  customer_id IS NOT NULL
)
SELECT
  *
FROM
  dim_customer
  """)

    


In [0]:
#writing to dim_customer

(dim_customer.write
  .format("delta")
  .mode("overwrite")
  .option("overwriteSchema", "true")
  .saveAsTable("workspace.superstore.gold_dim_customer"))




In [0]:
#product dimention
dim_product = spark.sql(""" WITH dim_product AS (      
SELECT DISTINCT
  product_id,
  product_name,
  category,
  sub_category
FROM
  silver_superstore_view
WHERE 
  product_id IS NOT NULL
)
SELECT
  *
FROM
  dim_product
  """)

In [0]:
#writing to dim_product

(dim_product.write
  .format("delta")
  .mode("overwrite")
  .option("overwriteSchema", "true")
  .saveAsTable("workspace.superstore.gold_dim_product"))


In [0]:
#date dimention
dim_date = spark.sql("""
WITH raw_dates AS (
    SELECT DISTINCT
        to_date(transaction_timestamp, 'M/d/yyyy') AS clean_date
    FROM 
        silver_superstore_view
    WHERE 
        transaction_timestamp IS NOT NULL
)
SELECT
    date_format(clean_date, 'yyyyMMdd') AS date_id,
    year(clean_date) AS year,
    month(clean_date) AS month,
    day(clean_date) AS day
FROM 
    raw_dates
WHERE 
    clean_date IS NOT NULL
ORDER BY 
    clean_date ASC
""")

In [0]:
#writing to dim_date

(dim_date.write
  .format("delta")
  .mode("overwrite")
  .option("overwriteSchema", "true")
  .saveAsTable("workspace.superstore.gold_dim_date"))


In [0]:
#fact_sales 

fact_sales = (spark.sql("""
    WITH fact_sales AS (
    SELECT 
        receipt_id,
        CAST(date_format(to_date(transaction_timestamp, 'M/d/yyyy'), 'yyyyMMdd') AS INT) AS date_id,
        postal_code,
        product_id,
        customer_id,
        'Online' AS channel,
        transaction_timestamp,
        quantity,
        sales_price,
        discount,
        profit
    FROM silver_superstore_view
    WHERE receipt_id IS NOT NULL
    )
    SELECT
        *
    FROM
        fact_sales
"""))

In [0]:
#writing to fact_sales

(fact_sales.write
  .format("delta")
  .mode("overwrite")
  .option("overwriteSchema", "true")
  .saveAsTable("workspace.superstore.gold_fact_sales"))


In [0]:
inventory_table = "workspace.superstore.gold_fact_inventory"

if not spark.catalog.tableExists(inventory_table):
    print("Creating fact_inventory table...")
    #fact_inventory
    fact_inventory=(spark.sql("""
        WITH superstore_latest_date AS (
            SELECT MAX(transaction_timestamp) AS max_date 
            FROM silver_superstore_view
        ),
        inventory_base AS (
            SELECT DISTINCT
                CAST(date_format((SELECT max_date FROM superstore_latest_date), 'yyyyMMdd') AS INT) AS date_id,
                product_id,
                category
            FROM silver_superstore_view
            WHERE product_id IS NOT NULL
        ),
        inventory_calculated AS (
            SELECT
                product_id,
                date_id,
                CASE 
                    WHEN category = 'Furniture' THEN 5         
                    WHEN category = 'Office Supplies' THEN 50  
                    WHEN category = 'Technology' THEN 15       
                    ELSE 20                                    
                END AS restock_limit,
                CAST(rand() * 150 AS INT) AS stock_quantity
            FROM inventory_base
        )
        SELECT
            product_id,
            restock_limit,
            stock_quantity,
            CASE 
                WHEN stock_quantity <= restock_limit THEN true 
                ELSE false 
            END AS needs_restock,
            date_id
        FROM
            inventory_calculated
    """))

    #writing to fact_inventory
    (fact_inventory.write 
        .format("delta") 
        .mode("overwrite") 
        .option("overwriteSchema", "true") 
        .saveAsTable("workspace.superstore.gold_fact_inventory"))

else:
    print("Restocking inventory table...")

    spark.sql("""
        MERGE INTO workspace.superstore.gold_fact_inventory AS target
        USING (
            SELECT 
                product_id,
                SUM(quantity) AS sold_today,
                CAST(date_format(MAX(transaction_timestamp), 'yyyyMMdd') AS INT) AS current_date_id
            FROM 
                workspace.superstore.gold_fact_sales
            WHERE 
                date_format(transaction_timestamp, 'yyyy-MM-dd') = (
                    SELECT date_format(MAX(transaction_timestamp), 'yyyy-MM-dd') 
                    FROM workspace.superstore.gold_fact_sales
                )
            GROUP BY 
                product_id
        ) AS source
        ON target.product_id = source.product_id
        
        WHEN MATCHED THEN
        UPDATE SET 
            
            target.stock_quantity = CASE 
                -- Lack of stock (Auto-Restock)
                WHEN (target.stock_quantity - source.sold_today) <= target.restock_limit 
                    THEN (target.stock_quantity - source.sold_today) + target.restock_limit + CAST(rand() * 100 AS INT)

                ELSE 
                    (target.stock_quantity - source.sold_today)
            END,
            
            -- date_id refresh
            target.date_id = source.current_date_id,
            
            -- needs_restock goes back to false
            target.needs_restock = false
    """)

    print("Restocking inventory table (Success)!")

In [0]:
%sql 
select * from workspace.superstore.gold_fact_inventory


In [0]:
%sql 
select * from workspace.superstore.gold_fact_sales

In [0]:
df_online = spark.read.table("workspace.superstore.gold_fact_sales").filter("channel != 'POS'")

fact_sales_pos = spark.sql("""
    SELECT 
        receipt_id, date_id, postal_code, product_id, customer_id,
        'POS' AS channel, transaction_timestamp, quantity,
        sales_price, discount, profit
    FROM workspace.superstore.fact_sales_staging_pos
    WHERE receipt_id IS NOT NULL
""")

df_final = df_online.unionByName(fact_sales_pos, allowMissingColumns=True)

(df_final.write
    .format("delta")
    .mode("overwrite") 
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.superstore.gold_fact_sales")
)

In [0]:
%sql
select count(*) from workspace.superstore.gold_fact_sales